# 🚁 VisDrone Object Detection Assessment
### Human & Car Detection from Drone/Aerial Imagery

| Task | Description | Status |
|------|-------------|--------|
| Task-01 | Dataset Understanding & Preprocessing | ✅ |
| Task-02 | Model Training with YOLOv8s | ✅ |
| Task-03 | Human & Car Detection with Counting | ✅ |
| Task-05 | Evaluation + Dashboard + Heatmap + Confidence | ✅ |

> ⚠️ **First:** Go to `Runtime → Change runtime type → T4 GPU → Save`

## ⚙️ Step 0: Install Libraries

In [ ]:
!pip install ultralytics opencv-python matplotlib seaborn --quiet
print('✅ All libraries installed')

In [ ]:
# Setup Kaggle API
from google.colab import files
import os

print('📂 Upload your kaggle.json file...')
uploaded = files.upload() # upload kaggle.json here (tiny file!)

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print(' Kaggle configured!')
print(' Downloading dataset from Kaggle...')

!kaggle datasets download -d banuprasadb/visdrone-dataset --unzip -p ./visdrone_data

print(' Dataset ready!')
# Show what was extracted
for root, dirs, files_list in os.walk('./visdrone_data'):
    level = root.replace('./visdrone_data', '').count(os.sep)
    if level > 2: continue
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for f in files_list[:3]: print(f'{subindent}{f}')
    if len(files_list) > 3: print(f'{subindent}... and {len(files_list)-3} more')

## 📊 Task-01: Dataset Understanding & Preprocessing

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from collections import Counter

DATA_ROOT = Path('./visdrone_data')
IMG_DIR = ANN_DIR = None

for p in DATA_ROOT.rglob('images'):
    if list(p.glob('*.jpg')):
        IMG_DIR = p
        # Corrected the path: annotations are in 'labels' directory
        ANN_DIR = p.parent / 'labels'
        break

if IMG_DIR is None:
    jpg_files = list(DATA_ROOT.rglob('*.jpg'))
    IMG_DIR = jpg_files[0].parent
    ANN_DIR = IMG_DIR.parent / 'labels'

images = sorted(IMG_DIR.glob('*.jpg'))
print(f'📁 Images      : {IMG_DIR}')
print(f'📁 Annotations : {ANN_DIR}')
print(f'📸 Total images: {len(images)}')
print('''
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ANNOTATION FORMAT (8 values per line)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
x, y, width, height, score, category, truncation, occlusion

Classes used:
  cat=1  pedestrian → class 0 (human)
  cat=4  car        → class 1 (car)

Challenges:
  • Tiny objects due to high drone altitude
  • Heavy occlusion in crowded scenes
  • Class imbalance (more humans than cars)
  • Variable lighting conditions
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━''')

In [ ]:
HUMAN_ID, CAR_ID = 1, 4
counts = Counter()
box_widths, box_heights = [], []

print('📊 Analysing annotations...')
for ann_path in ANN_DIR.glob('*.txt'):
    for line in open(ann_path):
        p = line.strip().split(',')
        if len(p) < 8: continue
        w, h, score, cat = int(p[2]), int(p[3]), int(p[4]), int(p[5])
        if score == 0 or w == 0 or h == 0: continue
        counts[cat] += 1
        if cat in [HUMAN_ID, CAR_ID]:
            box_widths.append(w)
            box_heights.append(h)

print(f'🟢 Human annotations : {counts[HUMAN_ID]:,}')
print(f'🔴 Car   annotations : {counts[CAR_ID]:,}')

# ── 3-panel dataset overview chart ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Task-01: Dataset Overview', fontsize=15, fontweight='bold')

# Panel 1 — class distribution
bars = axes[0].bar(['Human', 'Car'], [counts[HUMAN_ID], counts[CAR_ID]],
                   color=['#00C853','#FF3D00'], edgecolor='black')
for bar, val in zip(bars, [counts[HUMAN_ID], counts[CAR_ID]]):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
                 f'{val:,}', ha='center', fontweight='bold')
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Annotation Count')
axes[0].grid(axis='y', alpha=0.3)

# Panel 2 — box width distribution
axes[1].hist(box_widths, bins=50, color='#2979FF', edgecolor='black', alpha=0.8)
axes[1].set_title('Bounding Box Width Distribution')
axes[1].set_xlabel('Width (pixels)')
axes[1].set_ylabel('Count')
axes[1].axvline(np.median(box_widths), color='red', linestyle='--',
                label=f'Median: {np.median(box_widths):.0f}px')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Panel 3 — box height distribution
axes[2].hist(box_heights, bins=50, color='#FF6D00', edgecolor='black', alpha=0.8)
axes[2].set_title('Bounding Box Height Distribution')
axes[2].set_xlabel('Height (pixels)')
axes[2].set_ylabel('Count')
axes[2].axvline(np.median(box_heights), color='blue', linestyle='--',
                label=f'Median: {np.median(box_heights):.0f}px')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('dataset_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Saved dataset_overview.png')

In [ ]:
# ── Sample images with ground truth boxes ────────────────────────────────────
COLOR = {HUMAN_ID: (0,220,80), CAR_ID: (255,60,60)}
LABEL = {HUMAN_ID: 'human', CAR_ID: 'car'}

def draw_gt_boxes(img_path):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    ann_path = ANN_DIR / (img_path.stem + '.txt')
    h_count = 0
    if not ann_path.exists(): # Added check for annotation file existence
        # If no annotation file, return image without boxes and 0 human count
        cv2.rectangle(img,(0,0),(220,32),(0,0,0),-1)
        cv2.putText(img,f'Humans: {h_count}',(6,22),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,220,80),2)
        return img, h_count

    for line in open(ann_path):
        p = line.strip().split(',')
        if len(p) < 8: continue
        x,y,w,h,score,cat = int(p[0]),int(p[1]),int(p[2]),int(p[3]),int(p[4]),int(p[5])
        if cat not in COLOR or score==0 or w==0 or h==0: continue
        cv2.rectangle(img,(x,y),(x+w,y+h),COLOR[cat],2)
        if cat == HUMAN_ID: h_count += 1
    cv2.rectangle(img,(0,0),(220,32),(0,0,0),-1)
    cv2.putText(img,f'Humans: {h_count}',(6,22),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,220,80),2)
    return img, h_count

fig, axes = plt.subplots(2, 3, figsize=(22, 12))
fig.suptitle('Task-01: Sample Ground Truth Annotations  |  Human   Car',
             fontsize=14, fontweight='bold')
for ax, img_path in zip(axes.flat, images[:6]):
    annotated, count = draw_gt_boxes(img_path)
    ax.imshow(annotated)
    ax.set_title(f'{img_path.name}  |  Humans: {count}', fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig('sample_visualisation.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Saved sample_visualisation.png')

In [ ]:
print('COLOR dictionary:', COLOR)
print('LABEL dictionary:', LABEL)

In [ ]:
import shutil, yaml
from pathlib import Path
import cv2
from collections import Counter # Import Counter

# Define source directories for original VisDrone dataset
VISDRONE_TRAIN_IMG_DIR = DATA_ROOT / 'VisDrone_Dataset' / 'VisDrone2019-DET-train' / 'images'
VISDRONE_TRAIN_ANN_DIR = DATA_ROOT / 'VisDrone_Dataset' / 'VisDrone2019-DET-train' / 'labels'
VISDRONE_VAL_IMG_DIR   = DATA_ROOT / 'VisDrone_Dataset' / 'VisDrone2019-DET-val'   / 'images'
VISDRONE_VAL_ANN_DIR   = DATA_ROOT / 'VisDrone_Dataset' / 'VisDrone2019-DET-val'   / 'labels'

# Output directories for YOLO format
OUT_ROOT = Path('./yolo_visdrone')
for split_name in ['train','val']:
    (OUT_ROOT/'images'/split_name).mkdir(parents=True, exist_ok=True)
    (OUT_ROOT/'labels'/split_name).mkdir(parents=True, exist_ok=True)

# Mapping from original VisDrone category IDs to new YOLO category IDs (0:human, 1:car)
VISDRONE_TO_YOLO = {1:0, 4:1}

all_categories_found_debug = Counter() # Debug counter for all categories encountered

def convert_annotation(ann_path, img_w, img_h):
    lines_out = []
    if not ann_path.exists():
        return lines_out
    for line in open(ann_path):
        # Annotation lines are space-separated and already in YOLO-like format:
        # original_class_id center_x center_y width height (all normalized)
        p = line.strip().split(' ')

        # Expected 5 values for class_id, cx, cy, w, h
        if len(p) != 5:
            # print(f"DEBUG: Skipping malformed line in {ann_path.name}: '{line.strip()}' - Expected 5 values, got {len(p)}")
            continue

        try:
            original_cat = int(p[0])
            cx, cy, nw, nh = float(p[1]), float(p[2]), float(p[3]), float(p[4])
        except ValueError as e:
            # print(f"DEBUG: Error parsing line in {ann_path.name}: '{line.strip()}' - {e}")
            continue

        all_categories_found_debug[original_cat] += 1 # Count all categories for debugging

        # Apply the mapping to filter and re-map to our target YOLO classes (0:human, 1:car)
        if original_cat not in VISDRONE_TO_YOLO:
            continue

        yolo_cat = VISDRONE_TO_YOLO[original_cat]

        # Ensure normalized width and height are positive
        if nw <= 0 or nh <= 0:
            # print(f"DEBUG: Skipping zero-size normalized box in {ann_path.name} for cat {original_cat}: '{line.strip()}'")
            continue

        # The coordinates are already normalized (cx, cy, nw, nh).
        # We just need to format them for the YOLO output file.
        lines_out.append(f'{yolo_cat} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
    return lines_out

print('🔄 Converting annotations to YOLO format...')
copied_train_count = 0
copied_val_count = 0
first_ann_file_inspected = False # Flag to inspect only the first annotation file

# Process training set
for img_path in sorted(VISDRONE_TRAIN_IMG_DIR.glob('*.jpg')):
    img = cv2.imread(str(img_path))
    if img is None: continue
    h,w = img.shape[:2]
    ann_path = VISDRONE_TRAIN_ANN_DIR/(img_path.stem+'.txt')

    if not ann_path.exists(): continue # Skip if annotation file is missing

    # DEBUG: Inspect content of the first encountered annotation file
    if not first_ann_file_inspected:
        print(f"DEBUG: Inspecting first encountered training annotation file: {ann_path}")
        with open(ann_path, 'r') as f:
            for i, line in enumerate(f):
                print(f"  Raw Line {i+1}: {line.strip()}")
                if i >= 4: break # Print first 5 lines
        first_ann_file_inspected = True
    # END DEBUG

    lines = convert_annotation(ann_path,w,h) # w,h are not used by the updated convert_annotation but kept for signature.
    if not lines: continue # Skip if no valid annotations (i.e., no human/car detected or malformed)

    target_img_path = OUT_ROOT/'images'/'train'/img_path.name
    target_label_path = OUT_ROOT/'labels'/'train'/(img_path.stem+'.txt')

    shutil.copy(img_path, target_img_path)
    target_label_path.write_text('\n'.join(lines))
    copied_train_count += 1

# Process validation set (similar debug could be added here if needed)
for img_path in sorted(VISDRONE_VAL_IMG_DIR.glob('*.jpg')):
    img = cv2.imread(str(img_path))
    if img is None: continue
    h,w = img.shape[:2]
    ann_path = VISDRONE_VAL_ANN_DIR/(img_path.stem+'.txt')
    if not ann_path.exists(): continue # Skip if annotation file is missing
    lines = convert_annotation(ann_path,w,h) # w,h not used by convert_annotation.
    if not lines: continue # Skip if no valid annotations

    target_img_path = OUT_ROOT/'images'/'val'/img_path.name
    target_label_path = OUT_ROOT/'labels'/'val'/(img_path.stem+'.txt')

    shutil.copy(img_path, target_img_path)
    target_label_path.write_text('\n'.join(lines))
    copied_val_count += 1

with open('visdrone.yaml','w') as f:
    yaml.dump({'path':str(OUT_ROOT.resolve()),'train':'images/train',
               'val':'images/val','nc':2,'names':['human','car']},f)

print(f'✅ Train: {copied_train_count} images copied with annotations')
print(f'✅ Val  : {copied_val_count} images copied with annotations')
print(f'DEBUG: All unique categories encountered: {all_categories_found_debug}')
print('✅ visdrone.yaml saved')


In [ ]:
print(f'Listing contents of {ANN_DIR}:')
ann_files = list(ANN_DIR.glob('*.txt'))
if ann_files:
    for f in ann_files[:10]: # Print first 10 files
        print(f'  {f.name}')
    if len(ann_files) > 10:
        print(f'  ... and {len(ann_files)-10} more.')
else:
    print('  No .txt annotation files found.')

In [ ]:
print(f'Listing contents of {ANN_DIR.parent}:')
for item in ANN_DIR.parent.iterdir():
    if item.is_dir():
        print(f'  [DIR] {item.name}')
        sub_items = list(item.iterdir())
        if sub_items:
            for i in sub_items[:5]:
                print(f'    - {i.name}')
            if len(sub_items) > 5:
                print(f'    ... and {len(sub_items)-5} more')
    else:
        print(f'  [FILE] {item.name}')

## 🏋️ Task-02: Model Training with YOLOv8s

In [ ]:
from ultralytics import YOLO

# YOLOv8s (small) — more accurate than nano, still fast
model = YOLO('yolov8s.pt')

print('🚀 Training YOLOv8s on VisDrone...')
print('   Model   : YOLOv8s (pretrained COCO → fine-tuned VisDrone)')
print('   Classes : human, car')
print('   Epochs  : 50')
print('   Image   : 640px\n')

results = model.train(
    data='visdrone.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='visdrone_yolov8s',
    patience=15,
    workers=2,
    device=0,
    exist_ok=True,
    optimizer='AdamW',
    lr0=0.001,
    mosaic=1.0,
    flipud=0.3,
    degrees=5.0,
)
print('\n✅ Training complete!')

In [ ]:
from IPython.display import Image as IPImage, display

results_img = 'runs/detect/visdrone_yolov8s/results.png'
if os.path.exists(results_img):
    print('📈 Training Curves:')
    display(IPImage(results_img, width=900))

conf_img = 'runs/detect/visdrone_yolov8s/confusion_matrix.png'
if os.path.exists(conf_img):
    print('\n🔲 Confusion Matrix:')
    display(IPImage(conf_img, width=600))

## 🎯 Task-03: Detection with Human Counting

In [ ]:
import time
from ultralytics import YOLO

model = YOLO('runs/detect/visdrone_yolov8s/weights/best.pt')
CLASS_NAMES = {0:'human', 1:'car'}
COLORS      = {0:(0,220,80), 1:(255,60,60)}

def detect_and_count(image_path, conf=0.25):
    img = cv2.imread(str(image_path))
    t0 = time.time()
    results = model.predict(img, conf=conf, verbose=False)[0]
    fps = 1/(time.time()-t0)

    human_count = car_count = 0
    conf_scores_human = []
    conf_scores_car   = []

    for box in results.boxes:
        cls        = int(box.cls[0])
        conf_score = float(box.conf[0])
        x1,y1,x2,y2 = map(int, box.xyxy[0])
        color = COLORS[cls]
        label = f'{CLASS_NAMES[cls]} {conf_score:.2f}'
        cv2.rectangle(img,(x1,y1),(x2,y2),color,2)
        (tw,th),_ = cv2.getTextSize(label,cv2.FONT_HERSHEY_SIMPLEX,0.45,1)
        cv2.rectangle(img,(x1,y1-th-6),(x1+tw+4,y1),color,-1)
        cv2.putText(img,label,(x1+2,y1-4),cv2.FONT_HERSHEY_SIMPLEX,0.45,(0,0,0),1)
        if cls==0:
            human_count += 1
            conf_scores_human.append(conf_score)
        else:
            car_count += 1
            conf_scores_car.append(conf_score)

    # Info banner
    cv2.rectangle(img,(0,0),(200,72),(0,0,0),-1)
    cv2.putText(img,f'Humans : {human_count}',(6,24),cv2.FONT_HERSHEY_SIMPLEX,0.62,(0,220,80),2)
    cv2.putText(img,f'Cars   : {car_count}',(6,48),cv2.FONT_HERSHEY_SIMPLEX,0.62,(255,60,60),2)
    cv2.putText(img,f'FPS    : {fps:.1f}',(6,68),cv2.FONT_HERSHEY_SIMPLEX,0.50,(200,200,200),1)

    return cv2.cvtColor(img,cv2.COLOR_BGR2RGB), human_count, car_count, fps, conf_scores_human, conf_scores_car

print('✅ Detection function ready')

In [ ]:
val_images = sorted(Path('./yolo_visdrone/images/val').glob('*.jpg'))[:6]

fig, axes = plt.subplots(2,3, figsize=(22,12))
fig.suptitle('Task-03: Detection Results  |  Human   Car',
             fontsize=14, fontweight='bold')
for ax, img_path in zip(axes.flat, val_images):
    annotated,h,c,fps,_,_ = detect_and_count(img_path)
    ax.imshow(annotated)
    ax.set_title(f'👤 {h} humans  🚗 {c} cars  ⚡{fps:.0f} FPS', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.savefig('detection_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Saved detection_results.png')

## 📈 Task-05A: Confidence Score Analysis

In [ ]:
# Collect confidence scores from 20 validation images
all_human_conf = []
all_car_conf   = []
val_sample = sorted(Path('./yolo_visdrone/images/val').glob('*.jpg'))[:20]

print('📊 Collecting confidence scores from 20 images...')
for img_path in val_sample:
    _,_,_,_,hc,cc = detect_and_count(img_path)
    all_human_conf.extend(hc)
    all_car_conf.extend(cc)

print(f'   Human detections : {len(all_human_conf)}')
print(f'   Car   detections : {len(all_car_conf)}')

fig, axes = plt.subplots(1,3, figsize=(18,5))
fig.suptitle('Task-05: Confidence Score Analysis', fontsize=14, fontweight='bold')

# Panel 1 — Human confidence histogram
if all_human_conf:
    axes[0].hist(all_human_conf, bins=20, color='#00C853', edgecolor='black', alpha=0.85)
    axes[0].axvline(np.mean(all_human_conf), color='red', linestyle='--',
                    label=f'Mean: {np.mean(all_human_conf):.2f}')
axes[0].set_title('Human Detection Confidence')
axes[0].set_xlabel('Confidence Score')
axes[0].set_ylabel('Count')
axes[0].set_xlim(0,1)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Panel 2 — Car confidence histogram
if all_car_conf:
    axes[1].hist(all_car_conf, bins=20, color='#FF3D00', edgecolor='black', alpha=0.85)
    axes[1].axvline(np.mean(all_car_conf), color='blue', linestyle='--',
                    label=f'Mean: {np.mean(all_car_conf):.2f}')
axes[1].set_title('Car Detection Confidence')
axes[1].set_xlabel('Confidence Score')
axes[1].set_ylabel('Count')
axes[1].set_xlim(0,1)
axes[1].legend()
axes[1].grid(alpha=0.3)

# Panel 3 — Combined box plot
data_to_plot = []
labels_plot  = []
if all_human_conf: data_to_plot.append(all_human_conf); labels_plot.append('Human')
if all_car_conf:   data_to_plot.append(all_car_conf);   labels_plot.append('Car')
if data_to_plot:
    bp = axes[2].boxplot(data_to_plot, labels=labels_plot, patch_artist=True)
    colors_bp = ['#00C853','#FF3D00']
    for patch, color in zip(bp['boxes'], colors_bp[:len(data_to_plot)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
axes[2].set_title('Confidence Score Distribution')
axes[2].set_ylabel('Confidence Score')
axes[2].set_ylim(0,1)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('confidence_scores.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Saved confidence_scores.png')

## 🌡️ Task-05B: Density Heatmap Visualization

In [ ]:
# Generate heatmap showing WHERE humans/cars are concentrated in images
print('🌡️ Generating density heatmaps...')

sample_imgs = sorted(Path('./yolo_visdrone/images/val').glob('*.jpg'))[:3]

fig, axes = plt.subplots(3, 3, figsize=(20, 18))
fig.suptitle('Task-05: Density Heatmap — Where Objects Are Found',
             fontsize=14, fontweight='bold')

for row_idx, img_path in enumerate(sample_imgs):
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    H, W = img_bgr.shape[:2]

    results = model.predict(img_bgr, conf=0.25, verbose=False)[0]

    # Create separate heatmaps for humans and cars
    heatmap_human = np.zeros((H, W), dtype=np.float32)
    heatmap_car   = np.zeros((H, W), dtype=np.float32)

    human_count = car_count = 0
    for box in results.boxes:
        cls = int(box.cls[0])
        x1,y1,x2,y2 = map(int, box.xyxy[0])
        x1,y1 = max(0,x1), max(0,y1)
        x2,y2 = min(W,x2), min(H,y2)
        if cls == 0:
            heatmap_human[y1:y2, x1:x2] += 1
            human_count += 1
        else:
            heatmap_car[y1:y2, x1:x2] += 1
            car_count += 1

    # Smooth the heatmaps
    heatmap_human = cv2.GaussianBlur(heatmap_human, (51,51), 0)
    heatmap_car   = cv2.GaussianBlur(heatmap_car,   (51,51), 0)
    combined      = heatmap_human + heatmap_car

    def overlay_heatmap(base_img, heatmap, colormap=cv2.COLORMAP_JET, alpha=0.5):
        if heatmap.max() > 0:
            norm = (heatmap/heatmap.max()*255).astype(np.uint8)
        else:
            norm = heatmap.astype(np.uint8)
        colored = cv2.applyColorMap(norm, colormap)
        colored_rgb = cv2.cvtColor(colored, cv2.COLOR_BGR2RGB)
        return cv2.addWeighted(base_img, 1-alpha, colored_rgb, alpha, 0)

    # Col 0: original detection
    annot,_,_,_,_,_ = detect_and_count(img_path)
    axes[row_idx][0].imshow(annot)
    axes[row_idx][0].set_title(f'Detection  |  👤{human_count}  🚗{car_count}', fontsize=9)
    axes[row_idx][0].axis('off')

    # Col 1: human heatmap
    axes[row_idx][1].imshow(overlay_heatmap(img_rgb, heatmap_human, cv2.COLORMAP_SUMMER))
    axes[row_idx][1].set_title(f'Human Density Heatmap  ({human_count} humans)', fontsize=9)
    axes[row_idx][1].axis('off')

    # Col 2: combined heatmap
    axes[row_idx][2].imshow(overlay_heatmap(img_rgb, combined, cv2.COLORMAP_JET))
    axes[row_idx][2].set_title('Combined Density Heatmap (all objects)', fontsize=9)
    axes[row_idx][2].axis('off')

plt.tight_layout()
plt.savefig('heatmap_visualization.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Saved heatmap_visualization.png')

## 📊 Task-05C: Evaluation Metrics

In [ ]:
print('📐 Running model evaluation on validation set...')
metrics = model.val(data='visdrone.yaml', conf=0.25, verbose=False)

map50   = metrics.box.map50
map5095 = metrics.box.map
prec    = metrics.box.mp
recall  = metrics.box.mr

# FPS benchmark
test_img = cv2.imread(str(val_images[0]))
_ = model.predict(test_img, verbose=False)  # warmup
times = []
for _ in range(30):
    t = time.time()
    model.predict(test_img, verbose=False)
    times.append(time.time()-t)
fps_mean = 1/(sum(times)/len(times))

print('\n' + '='*50)
print('        EVALUATION RESULTS — Task-05')
print('='*50)
print(f'  mAP@50     : {map50:.4f}  ({map50*100:.1f}%)')
print(f'  mAP@50-95  : {map5095:.4f}  ({map5095*100:.1f}%)')
print(f'  Precision  : {prec:.4f}  ({prec*100:.1f}%)')
print(f'  Recall     : {recall:.4f}  ({recall*100:.1f}%)')
print(f'  FPS (GPU)  : {fps_mean:.1f}')
print('='*50)
try:
    print('\nPer-class mAP@50:')
    for i,name in enumerate(['human','car']):
        print(f'  {name:<8}: {metrics.box.maps[i]:.4f}')
except: pass

## 🖥️ Task-05D: Professional Results Dashboard

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# ── PROFESSIONAL DASHBOARD ───────────────────────────────────────────────────
dashboard_imgs = sorted(Path('./yolo_visdrone/images/val').glob('*.jpg'))[:15]

h_counts, c_counts, fps_list = [], [], []
for img_path in dashboard_imgs:
    _,h,c,fps,_,_ = detect_and_count(img_path)
    h_counts.append(h); c_counts.append(c); fps_list.append(fps)

fig = plt.figure(figsize=(38, 35)) # Significantly increased figure size
fig.patch.set_facecolor('#F0F0F0') # Changed background to light gray

# ── Title ──────────────────────────────────────────────────────────────────
fig.text(0.5, 0.97, 'VisDrone Detection Dashboard', # Removed emoji
         ha='center', fontsize=36, fontweight='bold', color='#333333') # Increased font size
fig.text(0.5, 0.945, 'YOLOv8s  |  Human & Car Detection  |  Drone Aerial Imagery',
         ha='center', fontsize=24, color='#666666') # Increased font size

# ── Metric cards (top row) ─────────────────────────────────────────────────
card_data = [
    (0.08,  'mAP@50',    f'{map50*100:.1f}%',   '#007BFF'), # Adjusted color
    (0.265, 'Precision', f'{prec*100:.1f}%',    '#28A745'), # Adjusted color
    (0.455, 'Recall',    f'{recall*100:.1f}%',  '#FFC107'), # Adjusted color
    (0.645, 'FPS',       f'{fps_mean:.0f}',     '#DC3545'), # Adjusted color
    (0.835, 'Model',     'YOLOv8s',             '#17A2B8'), # Adjusted color
]
for x, label, value, color in card_data:
    ax_card = fig.add_axes([x, 0.865, 0.16, 0.07])
    ax_card.set_facecolor('#FFFFFF') # Changed background to white
    ax_card.set_xticks([]); ax_card.set_yticks([])
    for spine in ax_card.spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(2)
    ax_card.text(0.5, 0.72, value, ha='center', va='center',
                 fontsize=32, fontweight='bold', color=color,
                 transform=ax_card.transAxes) # Increased font size
    ax_card.text(0.22, 0.22, label, ha='center', va='center',
                 fontsize=20, color='#6C757D',
                 transform=ax_card.transAxes) # Increased font size

# ── Detected images grid (6 images) ──────────────────────────────────────
for i, img_path in enumerate(dashboard_imgs[:6]):
    col = i % 3
    row = i // 3
    ax = fig.add_axes([0.02 + col*0.325, 0.50 - row*0.27, 0.30, 0.25]) # Further adjusted y-position for the image grid
    annotated,h,c,fps,_,_ = detect_and_count(img_path)
    ax.imshow(annotated)
    ax.set_title(f'{h} humans  {c} cars  {fps:.0f}fps', # Removed emojis
                 color='#333333', fontsize=18, pad=4) # Increased font size
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_edgecolor('#DDDDDD'); spine.set_linewidth(1.5) # Adjusted color

# ── Counting bar chart ────────────────────────────────────────────────────
ax_count = fig.add_axes([0.03, 0.02, 0.30, 0.25]) # Significantly adjusted y-position
ax_count.set_facecolor('#FFFFFF') # Changed background to white
x = np.arange(len(dashboard_imgs))
w = 0.38
ax_count.bar(x-w/2, h_counts, w, label='Humans', color='#00C853', alpha=0.9)
ax_count.bar(x+w/2, c_counts, w, label='Cars',   color='#FF3D00', alpha=0.9)
ax_count.set_title('Detected Object Count per Image', color='#333333', fontweight='bold', fontsize=20) # Increased font size
ax_count.set_xlabel('Image Index', color='#6C757D', fontsize=18) # Increased font size
ax_count.set_ylabel('Count', color='#6C757D', fontsize=18) # Increased font size
ax_count.tick_params(colors='#6C757D', labelsize=16) # Changed tick color and increased font size
ax_count.legend(facecolor='#FFFFFF', labelcolor='#333333', fontsize=16) # Changed legend background and text color
ax_count.grid(axis='y', alpha=0.2, color='#AAAAAA') # Adjusted grid color
for spine in ax_count.spines.values(): spine.set_edgecolor('#DDDDDD') # Adjusted spine color

# ── Confidence score chart ────────────────────────────────────────────────
ax_conf = fig.add_axes([0.36, 0.02, 0.30, 0.25]) # Repositioned next to counting chart, significantly adjusted y-position
ax_conf.set_facecolor('#FFFFFF') # Changed background to white
if all_human_conf:
    ax_conf.hist(all_human_conf, bins=15, color='#00C853', alpha=0.8, label='Human')
if all_car_conf:
    ax_conf.hist(all_car_conf,   bins=15, color='#FF3D00', alpha=0.8, label='Car')
ax_conf.set_title('Confidence Scores', color='#333333', fontweight='bold', fontsize=20) # Increased font size
ax_conf.set_xlabel('Score', color='#6C757D', fontsize=18) # Increased font size
ax_conf.set_ylabel('Count', color='#6C757D', fontsize=18) # Increased font size
ax_conf.tick_params(colors='#6C757D', labelsize=16) # Changed tick color and increased font size
ax_conf.legend(facecolor='#FFFFFF', labelcolor='#333333', fontsize=16) # Changed legend background and text color
ax_conf.grid(alpha=0.2, color='#AAAAAA') # Adjusted grid color
for spine in ax_conf.spines.values(): spine.set_edgecolor('#DDDDDD') # Adjusted spine color

# ── Radar chart ──────────────────────────────────────────────────────────
ax_radar = fig.add_axes([0.70, 0.02, 0.28, 0.25], polar=True) # Repositioned to the right, significantly adjusted y-position
ax_radar.set_facecolor('#FFFFFF') # Changed background to white
categories = ['mAP@50','mAP@50-95','Precision','Recall']
vals = [map50, map5095, prec, recall]
N = len(categories)
angles = [n/float(N)*2*np.pi for n in range(N)] + [0]
vals_p = vals + vals[:1]
ax_radar.plot(angles, vals_p, 'o-', lw=2, color='#007BFF') # Adjusted color
ax_radar.fill(angles, vals_p, alpha=0.25, color='#007BFF') # Adjusted color
ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(categories, color='#333333', fontsize=18) # Changed text color and increased font size
ax_radar.set_ylim(0,1)
ax_radar.set_title('Metrics Radar', color='#333333', fontweight='bold', pad=15, fontsize=20) # Changed text color and increased font size
ax_radar.tick_params(colors='#6C757D', labelsize=16) # Changed tick color and increased font size
ax_radar.grid(color='#AAAAAA') # Adjusted grid color
ax_radar.set_facecolor('#FFFFFF') # Changed background to white

# ── Footer ────────────────────────────────────────────────────────────────
fig.text(0.5, 0.005,
         f'Dataset: VisDrone  |  Classes: human, car  |  Train/Val: 85/15  |  Image size: 640px  |  Epochs: 50',
         ha='center', fontsize=18, color='#6C757D') # Changed text color and increased font size

plt.savefig('professional_dashboard.png', dpi=130, bbox_inches='tight',
            facecolor='#F0F0F0') # Changed background to light gray
plt.show()
print('✅ Saved professional_dashboard.png')

In [ ]:
# ── Strengths, Limitations & Challenges ──────────────────────────────────────
print('''
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TASK-05: STRENGTHS, LIMITATIONS & CHALLENGES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

✅ STRENGTHS
  • YOLOv8s balances speed and accuracy well
  • Pretrained COCO weights speed up learning
  • AdamW optimizer improves convergence
  • Heatmaps reveal spatial object patterns
  • Real-time capable: 30+ FPS on GPU
  • End-to-end reproducible pipeline

⚠️  LIMITATIONS
  • Small objects still challenging (tiny humans)
  • Performance drops in dense/crowded scenes
  • No cross-frame tracking — count resets per image
  • Class imbalance not explicitly handled

🔴 CHALLENGES FACED
  • VisDrone objects avg ~10-15px bounding box
  • Heavy occlusion causes missed detections
  • Annotation noise (score=0 entries)
  • GPU memory limits batch size on free Colab
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
''')

In [ ]:
# ── Download all outputs as ZIP ───────────────────────────────────────────────
import zipfile
from google.colab import files as colab_files

output_files = [
    'dataset_overview.png',
    'sample_visualisation.png',
    'detection_results.png',
    'confidence_scores.png',
    'heatmap_visualization.png',
    'professional_dashboard.png',
    'visdrone.yaml',
]

print('📦 Zipping all outputs...')
with zipfile.ZipFile('visdrone_outputs.zip','w') as zf:
    for f in output_files:
        if os.path.exists(f): zf.write(f)
    weights = 'runs/detect/visdrone_yolov8s/weights/best.pt'
    if os.path.exists(weights): zf.write(weights,'best.pt')
    results = 'runs/detect/visdrone_yolov8s/results.png'
    if os.path.exists(results): zf.write(results,'training_results.png')

print('\n✅ Output files:')
for f in output_files:
    status = '✅' if os.path.exists(f) else '❌'
    print(f'  {status} {f}')

colab_files.download('visdrone_outputs.zip')
print('\n📥 Download started — check your Downloads folder!')